# 02. AI Headline Generation (AR & AF)

This notebook generates AI-rewritten headlines for the **AR** and **AF** labels using the OpenAI Batch API.

## Label Definitions
| Label | Description |
|---|---|
| **AR** (AI-Real) | Human-real headline rewritten by GPT-4o-mini to sound more natural, while preserving all facts |
| **AF** (AI-Fake) | Fake headline generated by GPT-4o-mini from a real or fake seed, using one of six strategies |

## AF Generation Strategies
| Strategy | Source | Description |
|---|---|---|
| FS-1 | HR seed | Flip the main outcome / stance |
| FS-2 | HR seed | Change multiple key facts (large shift) |
| FS-3 | HR seed | Change one or two facts (small shift) |
| FB-1 | HF seed | Paraphrase — same claim, different wording |
| FB-2 | HF seed | Expand — add a short contextual phrase |
| FB-3 | HF seed | Disguise — change tone/style while keeping claim |

## Input
`HARFM_Step_1.csv` — output of `01_preprocessing.ipynb`

## Output
`HARFM.csv` — final dataset with `final_headline` column

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import time
from pathlib import Path
from openai import OpenAI
from tqdm.auto import tqdm

API_KEY = "YOUR_OPENAI_API_KEY"   # <-- replace with your key
client  = OpenAI(api_key=API_KEY)

DATA_DIR   = Path(".")
INPUT_CSV  = DATA_DIR / "HARFM_Step_1.csv"
CHUNK_SIZE = 40_000
SEED       = 42

---
## Step 2 — Generate AR Headlines

Rewrite each HR-seed headline to sound more natural while keeping all facts identical.

In [ ]:
BATCH_DIR_AR = DATA_DIR / "batch_requests_ar"
BATCH_DIR_AR.mkdir(exist_ok=True)

df_final     = pd.read_csv(INPUT_CSV)
df_ar_target = df_final[df_final["4_way_label"] == "AR"].copy()
print(f"AR target: {len(df_ar_target):,} rows")

AR_SYSTEM = "You are an AI writing assistant helping a Reddit user refine a news headline they wrote."
AR_USER   = """Task:
Rewrite the headline so it reads more natural and easy to understand as a Reddit-style news title,
while keeping all underlying facts exactly the same.

Writing rules:
- Keep every person, organization, location, date, and number from the original headline unchanged.
- Do not change the event itself or its outcome; only rephrase the wording and style.
- Make small but meaningful changes to wording and word order so the rewritten headline is not identical to the original.
- Try to keep the rewritten headline roughly similar in length to the original.
- Do not introduce any new facts, claims, or background information; use only what is in the original headline.

Original Reddit headline:
\"{title}\"

Rewritten Reddit headline (AI assistant):"""

for i in range(0, len(df_ar_target), CHUNK_SIZE):
    chunk     = df_ar_target.iloc[i : i + CHUNK_SIZE]
    file_path = BATCH_DIR_AR / f"ar_batch_{i//CHUNK_SIZE}.jsonl"
    with open(file_path, "w", encoding="utf-8") as f:
        for _, row in chunk.iterrows():
            task = {
                "custom_id": f"ar_{row['id']}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-mini",
                    "messages": [
                        {"role": "system", "content": AR_SYSTEM},
                        {"role": "user",   "content": AR_USER.format(title=row["clean_title"])}
                    ],
                    "temperature": 0.3
                }
            }
            f.write(json.dumps(task) + "\n")
    print(f"Created: {file_path}")

In [ ]:
# Upload & submit AR batches
job_tracker = []
for f_name in sorted(BATCH_DIR_AR.glob("*.jsonl")):
    print(f"Uploading: {f_name.name}")
    file_obj  = client.files.create(file=open(f_name, "rb"), purpose="batch")
    batch_job = client.batches.create(
        input_file_id=file_obj.id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )
    job_tracker.append({"file": f_name.name, "id": batch_job.id, "status": "enqueued"})
    print(f"  → Batch ID: {batch_job.id}")

# Poll until all done
print("\nPolling (every 10 min)...")
while True:
    active = 0
    for job in job_tracker:
        if job["status"] not in ["completed", "failed", "expired", "cancelled"]:
            res = client.batches.retrieve(job["id"])
            job["status"] = res.status
            if res.status == "completed":
                job["output_id"] = res.output_file_id
                print(f"Done: {job['file']}")
            elif res.status in ["failed", "expired", "cancelled"]:
                print(f"Failed: {job['file']} ({res.status})")
            else:
                active += 1
    if active == 0:
        print("All AR batches finished.")
        break
    time.sleep(600)

In [ ]:
# Download & merge AR results
if "ai_text" not in df_final.columns:
    df_final["ai_text"] = ""

results_dict = {}
for job in job_tracker:
    if job["status"] == "completed":
        for line in client.files.content(job["output_id"]).text.split("\n"):
            if not line.strip(): continue
            d  = json.loads(line)
            fid = str(d["custom_id"].split("_")[-1])
            try:
                results_dict[fid] = d["response"]["body"]["choices"][0]["message"]["content"].strip()
            except Exception as e:
                print(f"Parse error (id={fid}): {e}")

if df_final.index.name != "id" and "id" in df_final.columns:
    df_final.set_index("id", inplace=True)
df_final["ai_text"].update(pd.Series(results_dict).astype(df_final.index.dtype))
df_final.reset_index(inplace=True)

missing = ((df_final["4_way_label"] == "AR") & (df_final["ai_text"] == "")).sum()
print(f"AR merged: {len(results_dict):,} rows  |  missing: {missing}")

step2_path = DATA_DIR / "HARFM_Step_2_AR.csv"
df_final.to_csv(step2_path, index=False)
print(f"Saved: {step2_path}")

---
## Step 3 — Generate AF Headlines

Six strategies (FS-1/2/3 for HR seeds, FB-1/2/3 for HF seeds) are assigned in round-robin order.

In [ ]:
PROMPTS = {
    "HR": {
        "FS-1": """
[Task]
Rewrite the headline so the main outcome or stance is reversed,
while keeping the rest of the situation consistent with the original headline.

Writing rules:
- Keep every person, organization, location, date, and number from the original headline unchanged.
- Keep the same main actors and overall situation, but reverse the result, decision, or stance.
- Ensure the rewritten headline still reads like a natural Reddit-style news title.
- Keep the rewritten headline roughly similar in length to the original.
- Do not introduce unrelated new events or background details; use only what is in the original headline.

Original Reddit headline: "{title}"
Rewritten Reddit headline (flipped outcome, AI assistant): """,

        "FS-2": """
[Task]
Rewrite the headline by changing several factual details,
so that the situation becomes noticeably different but still realistic,
while keeping the general topic and type of event consistent with the original headline.

Writing rules:
- Keep the same overall event type and topic as in the original headline.
- Change multiple key factual elements, such as numbers, locations, people, organizations, or the scale of the event.
- Ensure all details in the rewritten headline are internally consistent with each other.
- Ensure the rewritten headline still reads like a natural and plausible Reddit-style news headline.
- Avoid adding extreme or sensational details that are not implied by the original headline.
- Keep the rewritten headline relatively short and similar in style and length to the original.

Original Reddit headline: "{title}"
Rewritten Reddit headline (large-shift, AI assistant): """,

        "FS-3": """
[Task]
Rewrite the headline by slightly changing one or two factual details,
so that the situation remains almost the same and the headline looks almost identical at a glance,
while keeping the overall event, topic, and framing consistent with the original headline.

Writing rules:
- Keep the same overall event, main actors, and topic as in the original headline.
- Change only one or two factual elements (such as a number, date, location, or scale of the event)
  so that the headline becomes subtly factually incorrect but still highly plausible.
- The rewritten headline should look almost identical to the original at a glance, with only small but meaningful differences.
- Ensure all details in the rewritten headline are internally consistent with each other.
- Ensure the rewritten headline still reads like a natural and plausible Reddit-style news headline.
- Avoid adding extreme or sensational details that are not implied by the original headline.
- Keep the rewritten headline relatively short and similar in style and length to the original.

Original Reddit headline: "{title}"
Rewritten Reddit headline (small-shift, AI assistant):"""
    },
    "HF": {
        "FB-1": """
[Task]
Rewrite the headline in different words, while keeping exactly the same underlying claim and meaning.

Writing rules:
- Keep all people, organizations, locations, dates, and numbers from the original headline unchanged.
- Keep the same underlying event and claim; do not add, remove, or alter any facts.
- Use different phrasing and word order so the rewritten headline is clearly paraphrased, not a trivial edit of the original.
- Keep the rewritten headline roughly similar in length and style to the original.

Original Reddit headline: "{title}"
Rewritten Reddit headline (AI assistant): """,

        "FB-2": """
[Task]
Expand the headline by adding a short phrase of framing or contextual detail,
while keeping the same underlying claim and meaning.

Writing rules:
- Preserve the core claim and what is being asserted; do not change or weaken it.
- Add one short phrase of background or framing detail (for example, a brief reason, time, place, or consequence)
  that makes the claim sound more concrete or specific.
- Do not introduce entirely new, unrelated claims.
- Keep the result as a single headline-style sentence (not a full paragraph).
- Keep the expanded headline only slightly longer than the original (within about 120% of the original length).
- Ensure the expanded headline still reads like a natural and plausible Reddit-style news title.

Original Reddit headline: "{title}"
Expanded Reddit headline (AI assistant): """,

        "FB-3": """
[Task]
Disguise the headline by changing its surface wording, sentence structure, and tone,
while keeping the same underlying claim and meaning.

Writing rules:
- Preserve the same core claim and overall meaning; do not change or replace it with a different claim.
- Do not change the main people, organizations, locations, or events involved.
- You may change sentence structure, level of formality, and tone
  so that the new headline feels stylistically different from the original and is harder to match to the original text.
- Use clearly different wording and style so the rewritten headline is not a trivial edit of the original.
- Keep the rewritten headline relatively short and natural, as if it could realistically appear online.

Original Reddit headline: "{title}"
Disguised Reddit headline (AI assistant):"""
    }
}

In [ ]:
BATCH_DIR_AF = DATA_DIR / "batch_requests_af"
BATCH_DIR_AF.mkdir(exist_ok=True)

df = pd.read_csv(DATA_DIR / "HARFM_Step_2_AR.csv")
df_af = df[df["4_way_label"] == "AF"].copy()

def assign_strategy(group):
    strategies = list(PROMPTS[group.name].keys())
    group["strategy"] = [strategies[i % 3] for i in range(len(group))]
    return group

df_af = df_af.groupby("ai_source", group_keys=False).apply(assign_strategy)
print(f"AF target: {len(df_af):,} rows")
print(df_af.groupby(["ai_source", "strategy"]).size())

AF_SYSTEM = "You are an AI writing assistant helping a Reddit user refine a Reddit news headline they wrote."

for i in range(0, len(df_af), CHUNK_SIZE):
    chunk     = df_af.iloc[i : i + CHUNK_SIZE]
    file_path = BATCH_DIR_AF / f"af_batch_{i//CHUNK_SIZE}.jsonl"
    with open(file_path, "w", encoding="utf-8") as f:
        for _, row in chunk.iterrows():
            strat, src = row["strategy"], row["ai_source"]
            task = {
                "custom_id": f"af_{strat}_{row['id']}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-mini",
                    "messages": [
                        {"role": "system", "content": AF_SYSTEM},
                        {"role": "user",   "content": PROMPTS[src][strat].format(title=row["clean_title"])}
                    ],
                    "temperature": 0.3
                }
            }
            f.write(json.dumps(task) + "\n")
    print(f"Created: {file_path}")

In [ ]:
# Upload & submit AF batches
batch_jobs_af = []
for bf in sorted(BATCH_DIR_AF.glob("af_batch_*.jsonl")):
    print(f"Uploading: {bf.name}")
    file_obj  = client.files.create(file=open(bf, "rb"), purpose="batch")
    batch_job = client.batches.create(
        input_file_id=file_obj.id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )
    batch_jobs_af.append(batch_job.id)
    print(f"  → Batch ID: {batch_job.id}")

# Poll
output_ids_af = {}
while len(output_ids_af) < len(batch_jobs_af):
    for bid in batch_jobs_af:
        if bid in output_ids_af: continue
        status = client.batches.retrieve(bid)
        done, total = status.request_counts.completed, status.request_counts.total
        print(f"  {bid}: {status.status} ({done}/{total})")
        if status.status == "completed":
            output_ids_af[bid] = status.output_file_id
        elif status.status in ["failed", "expired", "cancelled"]:
            raise Exception(f"Batch {bid} failed: {status.status}")
    if len(output_ids_af) < len(batch_jobs_af):
        time.sleep(30)
print("All AF batches finished.")

In [ ]:
# Download & merge AF results
af_results, af_strategies = {}, {}
for batch_id, file_id in output_ids_af.items():
    for line in client.files.content(file_id).text.split("\n"):
        if not line.strip(): continue
        d = json.loads(line)
        parts = d["custom_id"].split("_")
        if len(parts) >= 3:
            strat = parts[1]
            oid   = parts[-1]
            try:
                af_results[oid]    = d["response"]["body"]["choices"][0]["message"]["content"].strip()
                af_strategies[oid] = strat
            except Exception as e:
                print(f"Parse error (id={oid}): {e}")

if "ai_strategy" not in df.columns:
    df["ai_strategy"] = ""
if df.index.name != "id" and "id" in df.columns:
    df.set_index("id", inplace=True)
idx_dtype = df.index.dtype
df["ai_text"].update(pd.Series(af_results).astype(idx_dtype))
df["ai_strategy"].update(pd.Series(af_strategies).astype(idx_dtype))
df.reset_index(inplace=True)
print(f"AF merged: {len(af_results):,} rows")

In [ ]:
# Build final_headline column and save
df["final_headline"] = df["clean_title"]
df.loc[df["4_way_label"].isin(["AR", "AF"]), "final_headline"] = df["ai_text"]

final_path = DATA_DIR / "HARFM.csv"
df.to_csv(final_path, index=False, encoding="utf-8-sig")
print(f"Saved: {final_path}  ({len(df):,} rows)")
print(df["4_way_label"].value_counts())